# 2026 지옥캠프: 6공학관 vs IT관 분류 — 베이스라인 (Small CNN)

약 40만 파라미터의 소형 CNN으로, 끝까지 실행하면 제출용 CSV와 모델 패키지가 생성되고 그럴듯한 점수가 나오는 출발점입니다.

- 작성 환경: Google Colab (CPU만으로도 동작) / 로컬 Python 3.10+, PyTorch 2.x
- GPU가 있으면 자동으로 사용하지만 **필수는 아닙니다**

## 대회 규칙 요약 (필독)

| 항목 | 내용 |
|---|---|
| 과제 | 사진 → **6공학관(0) / IT관(1)** 이진 분류 |
| 참가 | 개인전 |
| 기간 | 2026-11-07(토) ~ 11-08(일) |
| 지표 | **Log Loss** (낮을수록 우수, 확률을 제출해야 함) |
| 모델 제약 | **총 파라미터 20M 이하** (앙상블·사전학습 백본 합산) |
| 런타임 | **Colab CPU 기준** 테스트 500장 10분 이내 · 실행 중 다운로드 금지 (사전학습 가중치는 최종 파일에 포함) |
| 금지 | 외부 이미지 사용, 테스트 이미지를 학습에 사용(pseudo-labeling 포함) |
| 제출 | 예측 CSV **10분에 1회 · 하루 최대 50회** (공개 리더보드) + 마감 시 최종 파일 1개 (형식 자유) |
| 순위 | 테스트 500장 중 Public 250장 = 공개 리더보드, **Private 250장 = 최종 순위** (어느 장이 Private 인지는 비공개) |

### 이 제출본의 실제 모델
위 설명과 목차는 제공된 원본을 보존한 것입니다. 아래 코드는 seed 17의 최종 모델에 맞춰 옮겼습니다.
- 모델: ImageNet 사전학습 EfficientNet-B0 4개, 합계 16,035,316개 파라미터.
- 분할 seed 17, 모델 seed 17·18·19·20. 입력 192×192, 중앙 크롭 추론.
- 최종 예측: 모델별 원본·좌우 반전 확률 평균 → 조건부 T0.3 결합.
- 기존 model.pt와 submission_submit.csv는 Public 0.0066을 받은 산출물입니다.
- 현재 노트북은 원본 학습 코드의 재배치본입니다. training_record.json의 노트북 해시는 실제 학습 당시 원본을 가리킵니다.


## 실행 전 사전 준비

### 모듈
Colab은 필요한 모듈이 모두 설치되어 있습니다. 로컬이라면:
```shell
# 본인 환경(CPU/CUDA)에 맞는 PyTorch 설치: https://pytorch.org/get-started/locally/
pip3 install torch torchvision
pip3 install -U scikit-learn matplotlib tqdm ipywidgets
```

### 데이터 파일
운영진이 배포한 데이터를 노트북과 같은 폴더에 압축해제해 주세요 (Colab 은 `/content`).
```
📁root
 ├─📁train
 │  ├─📁6e      ← 6공학관 (label 0)
 │  └─📁it      ← IT관   (label 1)
 ├─📁test       ← 테스트 500장 (Public 250 + Private 250)
 └─📜baseline.ipynb
```

### 현재 파일의 실행 안내
실행 파일은 baseline_new.ipynb입니다. 재학습은 별도 빈 작업 폴더에 이 노트북과 train/test/sample_submission.csv를 준비한 뒤 전체 실행합니다.
새 가중치·CSV·패키지 폴더는 RESULT_DIR 아래에 저장하며 기존 최종 model.pt와 submission_submit.csv를 덮어쓰지 않습니다.
학습에 필요한 모듈은 numpy, torch, torchvision, scikit-learn, matplotlib, Pillow입니다.
제공된 가중치로 추론만 할 때는 노트북 재학습 없이 README.txt의 predict.py 명령을 사용합니다.
같은 학습 결과 비교는 동일한 Python·라이브러리·장치 설정을 전제로 하며, 다른 환경에서의 비트 단위 일치는 보장되지 않습니다.


## 모듈 불러오기 & 시드 고정

재현성을 위해 모든 난수 시드를 고정합니다. 시드가 고정되어 있어야 "내가 바꾼 것 때문에 점수가 오른 것인지, 운이 좋았던 것인지"를 구분할 수 있습니다.

In [ ]:
import csv
import importlib
import os
import shutil
import subprocess
import sys
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 17
BASE_DIR = Path.cwd().resolve()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(SEED)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
	print(torch.cuda.get_device_name())


## 하이퍼 파라미터

In [ ]:
params = {
	'epoch': 50,
	'early_stopping': True,
	'min_epochs': 5,
	'patience': 5,
	'batch_size': 32,
	'learning_rates': (0.001, 0.0003, 0.0001),
	'valid_ratio': 0.15,
	'num_show_img': 5,
}


In [ ]:
class_names = {0: '6e', 1: 'it'}
RUN_SEEDS = tuple(range(SEED, SEED + 4))
AUGMENTATION_BY_SEED = dict(zip(RUN_SEEDS, ('square', 'rectangle', 'square_erasing', 'square_perspective')))
RESULT_DIR = BASE_DIR / '실험결과' / '011_192크롭_패딩제거' / f'seed_{SEED}'
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
EPOCH_DIR = RESULT_DIR / 'epochs'
assert not (RESULT_DIR / 'runs.csv').exists(), f'기존 학습 결과를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not (RESULT_DIR / 'model.pt').exists(), f'기존 가중치를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not list(CHECKPOINT_DIR.glob('*.pt')), f'체크포인트가 이미 있습니다: {CHECKPOINT_DIR}'
for directory in (CHECKPOINT_DIR, EPOCH_DIR):
	directory.mkdir(parents=True, exist_ok=True)
print('결과 위치:', RESULT_DIR)


## 모델 정의 (`model.py` 생성)

아래 셀은 `%%writefile` 매직으로 **`model.py` 파일을 생성**합니다. 이 파일은 나중에 모델 패키지에 그대로 들어가므로, 모델 구조는 반드시 이 셀에서 수정하세요. 수정 후에는 셀을 다시 실행해야 파일에 반영됩니다.

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as F

IMG_SIZE = 192
MIN_CROP_SIZE = 96
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PERSPECTIVE_FILL = tuple(round(value * 255) for value in NORM_MEAN)
TEMPERATURE_RULES = ("T0.3", "T0.5")


class RandomPhotoCrop:
	def __init__(self, square):
		self.square = square

	def __call__(self, image):
		width, height = image.size
		assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작은 원본입니다: {image.size}"
		crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		if self.square:
			crop_height = crop_width
		else:
			crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
			while (crop_width / crop_height < 0.75) or (crop_width / crop_height > 4 / 3):
				crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
				crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		top = int(torch.randint(height - crop_height + 1, ()).item())
		left = int(torch.randint(width - crop_width + 1, ()).item())
		return F.crop(image, top, left, crop_height, crop_width)


def center_crop(image):
	width, height = image.size
	assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192より小さいためパディングなしで切り出せません: {image.size}"
	return F.center_crop(image, [IMG_SIZE, IMG_SIZE])


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(center_crop),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs, temperature_rule="T0.3"):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 4)
	assert bool(torch.isfinite(member_probs).all()), "モデル確率に非有限値があります。"
	assert bool(((member_probs >= 0) & (member_probs <= 1)).all()), "モデル確率が範囲外です。"
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	middle_mean = member_probs.sort(dim=1).values[:, 1:3].mean(dim=1, keepdim=True)
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), middle_mean)
	if temperature_rule == "T0.3":
		temperatures = torch.where(unanimous, 0.3, 0.7)
	elif temperature_rule == "T0.5":
		temperatures = 0.5
	else:
		raise ValueError(f"알 수 없는 temperature 규칙: {temperature_rule}")
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(pretrained=False):
	weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
	net = models.efficientnet_b0(weights=weights)
	net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	return net


class EfficientNetEnsemble(nn.Module):
	def __init__(self, state_dicts):
		super().__init__()
		assert len(state_dicts) == 4, "앙상블 가중치 4개가 필요합니다."
		self.models = nn.ModuleList()
		for state_dict in state_dicts:
			model = build_model()
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs):
		return torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)

	def forward(self, inputs, temperature_rule="T0.3"):
		return combine_probabilities(self.member_probabilities(inputs), temperature_rule)


In [ ]:
import model as model_module
importlib.invalidate_caches()
importlib.reload(model_module)
assert Path(model_module.__file__).resolve() == BASE_DIR / 'model.py'
model = model_module.build_model()
print('EfficientNet-B0 / input:', model_module.IMG_SIZE)


### 파라미터 카운터 (20M 제한 확인)

앙상블을 하거나 사전학습 백본을 쓰는 경우 **모든 모델의 파라미터를 합산**한 값이 20,000,000 이하여야 합니다.

현재 파라미터 검사는 단일 모델뿐 아니라 최종 네 모델 합산을 검사합니다.


In [ ]:
PARAM_LIMIT = 20_000_000
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f'1모델 {n_params:,} / 4모델 {4 * n_params:,} / 제한 {PARAM_LIMIT:,}')
assert 4 * n_params <= PARAM_LIMIT
del model


## 데이터 불러오기 & 증강

- 학습(train) 데이터에만 증강을 적용하고, 검증(valid) 데이터는 증강 없이 평가합니다.
- 두 transform 모두 마지막에 채널별 **정규화(Normalize)** 를 적용합니다. 픽셀 분포를 평균 0 근처로 맞추면 학습이 안정됩니다.
- 증강은 자유입니다 (flip·crop·color·noise·mixup 등). 다양하게 실험해 보세요.
- **주의**: 최종 순위는 공개 리더보드에 보이지 않는 Private 250장으로 정합니다. 공개 점수에 맞춰 튜닝하기보다 밝기·색감·화각 변화에 강한 모델을 만드는 것이 최종 점수에 유리합니다.

현재 학습은 실제 사진에서 각 변 96~192픽셀을 크롭하고 192×192로 확대합니다. 검증·추론은 192×192 중앙 크롭입니다. 크기 맞춤 패딩은 없으며 원근 변환의 모서리 채움은 유지합니다.


In [ ]:
data_path = BASE_DIR / 'train'
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD
AUGMENTATION_SETTINGS = {
	'square': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.0},
	'rectangle': {'square': False, 'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.3, 'erase_p': 0.0, 'perspective_p': 0.0},
	'square_erasing': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.1, 'perspective_p': 0.0},
	'square_perspective': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.3},
}
train_transforms = {}
for name, settings in AUGMENTATION_SETTINGS.items():
	operations = [
		model_module.RandomPhotoCrop(square=settings['square']),
		transforms.RandomHorizontalFlip(p=0.5),
		transforms.ColorJitter(brightness=settings['brightness'], contrast=settings['contrast'], saturation=settings['saturation']),
	]
	if settings['perspective_p'] > 0:
		operations.append(transforms.RandomPerspective(distortion_scale=0.15, p=settings['perspective_p'], fill=model_module.PERSPECTIVE_FILL))
	operations.extend([
		transforms.Resize((IMG_SIZE, IMG_SIZE)),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])
	if settings['erase_p'] > 0:
		operations.append(transforms.RandomErasing(p=settings['erase_p'], scale=(0.02, 0.1), value=0))
	train_transforms[name] = transforms.Compose(operations)
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transforms['square'])
full_valid = datasets.ImageFolder(data_path, valid_transform)
assert full_train.class_to_idx == {'6e': 0, 'it': 1}
assert len(full_train) == 2906, '학습' + f' 사진 수가 다릅니다: {len(full_train)}'
for path, label in full_train.samples:
	with Image.open(path) as image:
		assert min(image.size) >= IMG_SIZE, f'192보다 작은 이미지: {path}'
print('클래스:', full_train.class_to_idx, '/ 전체:', len(full_train), '/ 입력:', IMG_SIZE)


### Train / Valid 분할

클래스 비율이 유지되도록 층화(stratified) 분할합니다. 검증 점수(log loss)가 리더보드 점수의 예측치 역할을 하므로, **검증 데이터를 학습에 섞지 않도록** 주의하세요.

In [ ]:
labels = [label for _, label in full_train.samples]
train_idx, valid_idx = train_test_split(list(range(len(full_train))), test_size=params['valid_ratio'], stratify=labels, random_state=SEED)
assert (len(train_idx) == 2470) and (len(valid_idx) == 436)
assert not set(train_idx).intersection(valid_idx)
data_sets = {'train': Subset(full_train, train_idx), 'valid': Subset(full_valid, valid_idx)}
with (RESULT_DIR / 'validation_split.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'label'))
	writer.writerows((Path(full_valid.samples[index][0]).stem, labels[index]) for index in valid_idx)
print(f'train: {len(train_idx)} / valid: {len(valid_idx)} / split seed: {SEED}')


In [ ]:
dataloaders = {
	'train': DataLoader(data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED)),
	'valid': DataLoader(data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
}
print('배치 수:', len(dataloaders['train']), len(dataloaders['valid']))


### 데이터 확인

아래 미리보기 이후 각 모델 학습 직전에 seed를 다시 고정하므로 미리보기가 학습 난수 흐름을 바꾸지 않습니다.


In [ ]:
def img_show(tensor, title=None):
	arr = tensor.numpy().transpose((1, 2, 0))
	arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
	arr = np.clip(arr, 0, 1)
	plt.figure(figsize=(12, 4))
	plt.imshow(arr)
	plt.axis('off')
	if title is not None:
		plt.title(title)
	plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])


## 학습

손실함수는 `BCEWithLogitsLoss`를 사용합니다. 이 값이 곧 대회 지표인 **Log Loss와 동일**하므로, 검증 loss를 그대로 "내 예상 리더보드 점수"로 읽으면 됩니다.

검증 log loss가 가장 낮았던 epoch의 가중치를 `model.pt`로 저장합니다.

현재 코드는 네 모델을 각각 학습합니다. 검증 손실은 BCEWithLogitsLoss로 체크포인트와 학습률 전환을 결정합니다.
검증 손실은 최종 TTA·temperature 적용 후 Log Loss 및 비공개 Public 점수와 구분해야 합니다.
단계별 학습률 0.001 → 0.0003 → 0.0001, 최소 5 epoch·미개선 5회, 최대 50 epoch 규칙을 그대로 사용합니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
	"""optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
	training = optimizer is not None
	model.train() if training else model.eval()

	total_loss, total_correct, total_n = 0.0, 0, 0
	with torch.set_grad_enabled(training):
		for inputs, targets in loader:
			inputs = inputs.to(device)
			targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

			logits = model(inputs)
			loss = criterion(logits, targets)
			assert bool(torch.isfinite(loss)), "학습/검증 손실이 비유한 값입니다."

			if training:
				optimizer.zero_grad()
				loss.backward()
				optimizer.step()

			probs = torch.sigmoid(logits)
			preds = (probs >= 0.5).float()
			total_loss += loss.item() * inputs.size(0)
			total_correct += (preds == targets).sum().item()
			total_n += inputs.size(0)

	return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates):
	assert min_epochs > 0
	assert patience > 0
	assert len(learning_rates) > 0
	assert all(rate > 0 for rate in learning_rates)
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}', flush=True)

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
results = []
histories = {}


In [ ]:
for seed in RUN_SEEDS:
	augmentation = AUGMENTATION_BY_SEED[seed]
	full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
	assert full_run_train.samples == full_train.samples
	run_data_sets = {'train': Subset(full_run_train, train_idx), 'valid': data_sets['valid']}
	print(f'\n학습 시작: seed {seed} / {augmentation}', flush=True)
	seed_everything(seed)
	model = model_module.build_model(pretrained=True).to(device)
	seed_everything(seed)
	dataloaders = {
		'train': DataLoader(run_data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed)),
		'valid': DataLoader(run_data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
	}
	criterion = nn.BCEWithLogitsLoss()
	optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rates'][0])
	pt_file_name = CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt'
	assert not pt_file_name.exists(), f'기존 가중치를 덮어쓰지 않습니다: {pt_file_name}'
	started = time.perf_counter()
	with (EPOCH_DIR / f'efficientnet_b0_seed{seed}.csv').open('w', newline='', encoding='utf-8-sig') as epoch_csv:
		history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params['epoch'], epoch_csv=epoch_csv, early_stopping=params['early_stopping'], min_epochs=params['min_epochs'], patience=params['patience'], learning_rates=params['learning_rates'])
	elapsed = time.perf_counter() - started
	histories[seed] = (history, best_epoch, best_loss)
	results.append({
		'model': 'efficientnet_b0', 'seed': seed, 'augmentation': augmentation,
		'image_size': IMG_SIZE, 'selection_metric': 'valid_loss', 'learning_rates': str(params['learning_rates']),
		'max_epochs': params['epoch'], 'epochs_run': len(history['train_loss']),
		'early_stopping': params['early_stopping'], 'min_epochs': params['min_epochs'], 'patience': params['patience'],
		'best_epoch': best_epoch, 'valid_loss': best_loss, 'valid_acc': history['valid_acc'][best_epoch - 1],
		'seconds': elapsed, 'n_params': n_params, 'checkpoint': str(pt_file_name),
	})
	with (RESULT_DIR / 'runs.csv').open('w', newline='', encoding='utf-8-sig') as output:
		writer = csv.DictWriter(output, fieldnames=list(results[0]))
		writer.writeheader()
		writer.writerows(results)
	del model, optimizer, dataloaders
	if device.type == 'cuda':
		torch.cuda.empty_cache()
assert len(results) == 4


### 학습 곡선

train과 valid의 간격이 크게 벌어지면 과적합 신호입니다.

In [ ]:
for seed in RUN_SEEDS:
	history, best_epoch, best_loss = histories[seed]
	epochs = range(1, len(history['train_loss']) + 1)
	fig, axes = plt.subplots(1, 2, figsize=(12, 4))
	for axis, metric in zip(axes, ('loss', 'acc')):
		axis.plot(epochs, history[f'train_{metric}'], label='train')
		axis.plot(epochs, history[f'valid_{metric}'], label='valid')
		axis.set_xlabel('epoch')
		axis.set_ylabel(metric)
		axis.legend()
		axis.grid(alpha=0.3)
	fig.suptitle(f'seed {seed} / selected epoch {best_epoch} / loss {best_loss:.6f}')
	fig.tight_layout()
	plt.show()
	plt.close(fig)


## 테스트 추론 → `submission.csv` 생성

리더보드 제출 형식은 `id,prob` 입니다. `prob`은 **IT관(=1)일 확률**입니다. 제출은 **10분에 1회, 하루 최대 50회**입니다.

```
id,prob
f81d4fae-7dec-41d0-a765-00a0c91e6bf6,0.831250
c9b1f2e8-3a45-4d67-b012-5e6f7a8b9c0d,0.024113
```

> 채점 시 확률은 [1e-6, 1-1e-6]로 클리핑되지만, 0이나 1에 극단적으로 붙은 확신은 한 장만 틀려도 log loss를 크게 망가뜨립니다. 모델이 과확신하지 않는지 확인해 보세요.

현재 생성되는 CSV는 원본·좌우 반전 TTA와 조건부 T0.3을 사용합니다. RESULT_DIR/submission.csv에 저장됩니다. 각 모델의 확률을 먼저 평균한 뒤 temperature는 결합 확률에 한 번 적용합니다.


In [ ]:
class TestImageDataset(Dataset):
	def __init__(self, files, transform):
		self.files = files
		self.transform = transform

	def __len__(self):
		return len(self.files)

	def __getitem__(self, index):
		path = self.files[index]
		with Image.open(path) as image:
			return self.transform(image.convert('RGB')), path.stem


test_files = sorted(path for path in (BASE_DIR / 'test').iterdir() if path.is_file() and (path.suffix.lower() in ('.jpg', '.jpeg', '.png')))
with (BASE_DIR / 'sample_submission.csv').open(newline='', encoding='utf-8-sig') as source:
	sample_rows = list(csv.DictReader(source))
expected_ids = {row['id'] for row in sample_rows}
assert (len(test_files) == 500) and (len(expected_ids) == 500)
assert len({path.stem for path in test_files}) == 500
assert {path.stem for path in test_files} == expected_ids
test_loader = DataLoader(TestImageDataset(test_files, valid_transform), batch_size=params['batch_size'], shuffle=False, num_workers=0)


In [ ]:
checkpoint = {
	'split_seed': SEED, 'image_size': IMG_SIZE, 'preprocessing': 'center_crop_192_no_padding',
	'seeds': list(RUN_SEEDS), 'augmentations': [AUGMENTATION_BY_SEED[seed] for seed in RUN_SEEDS],
	'state_dicts': [torch.load(CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt', map_location='cpu', weights_only=True) for seed in RUN_SEEDS],
}
torch.save(checkpoint, RESULT_DIR / 'model.pt')
ensemble = model_module.EfficientNetEnsemble(checkpoint['state_dicts']).eval()
ensemble_params = sum(parameter.numel() for parameter in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT
print('모델 저장:', RESULT_DIR / 'model.pt')

rows = []
with torch.inference_mode():
	for inputs, image_ids in test_loader:
		members = ensemble.member_probabilities(inputs)
		flipped_members = ensemble.member_probabilities(inputs.flip(dims=[3]))
		probs = model_module.combine_probabilities((members + flipped_members) / 2, 'T0.3').squeeze(1)
		assert bool(torch.isfinite(probs).all())
		rows.extend((image_id, f'{prob.item():.6f}') for image_id, prob in zip(image_ids, probs))
assert (len(rows) == 500) and ({row[0] for row in rows} == expected_ids)
output_path = RESULT_DIR / 'submission.csv'
with output_path.open('w', newline='', encoding='utf-8') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'prob'))
	writer.writerows(rows)
print('예측 저장:', output_path)


## 최종 파일 생성 (예시: `predict.py` 패키지)

마감 후 운영진이 **모든 참가자의 최종 파일**을 받아 **Colab CPU** 환경에서 직접 실행합니다.

1. 공개 리더보드(Public) 점수가 재현되는지 확인
2. 런타임(테스트 500장 10분 이내)과 파라미터 수(20M 이하) 확인
3. 통과한 참가자만 Private 250장 점수로 최종 순위 산정

**제출 형식은 자유**입니다. 운영진이 실행할 수 있고 가중치가 포함되어 있으면 됩니다. 아래는 이 노트북이 만드는 예시 형식이며, 이렇게 내면 다음 명령 한 줄로 검증됩니다.

```shell
python predict.py --input <이미지 폴더> --output pred.csv
```

패키지 구성:
```
📦submission_package.zip
 ├─📜model.py     # 모델 구조 정의 (build_model 함수 필수)
 ├─📜predict.py   # 추론 진입점 (--input/--output 인터페이스 유지)
 └─📜model.pt     # 학습된 가중치 (사전학습 가중치 사용 시에도 여기에 포함)
```

### 이 버전의 패키지 구성
새 압축 파일은 생성하지 않습니다. RESULT_DIR/package/에 model.py, predict.py, model.pt, baseline_new.ipynb를 복사합니다.
기본 predict.py 명령은 추가 옵션 없이 T0.3 TTA를 사용합니다. 추론에서는 pretrained=False로 모델 구조를 만들고 동봉된 전체 가중치를 불러오므로 다운로드하지 않습니다.
학습 노트북은 학습 재현용이며 평가 시에는 predict.py만 실행하면 됩니다.


In [ ]:
%%writefile predict.py
import argparse
import csv
from pathlib import Path

import torch
from PIL import Image

from model import IMG_SIZE, TEMPERATURE_RULES, EfficientNetEnsemble, build_valid_transform, combine_probabilities

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
	files = sorted(path for path in Path(input_dir).iterdir() if path.is_file() and (path.suffix.lower() in (".jpg", ".jpeg", ".png")))
	assert files, f"이미지가 없습니다: {input_dir}"
	assert len({path.stem for path in files}) == len(files), "이미지 ID가 중복됩니다."
	return files


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--temperature-rule", choices=TEMPERATURE_RULES, default="T0.3")
	parser.add_argument("--tta", action="store_true", default=True, help="모델별 원본·좌우 반전 확률 평균 후 결합")
	args = parser.parse_args()
	weights_path = Path(args.weights)
	if not weights_path.is_absolute():
		weights_path = Path(__file__).resolve().parent / weights_path
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE
	assert checkpoint["preprocessing"] == "center_crop_192_no_padding"
	assert len(checkpoint["seeds"]) == 4, "모델 seed 4개가 필요합니다."
	assert all(isinstance(seed, int) for seed in checkpoint["seeds"]), "모델 seed는 정수여야 합니다."
	assert checkpoint["seeds"] == list(range(checkpoint["seeds"][0], checkpoint["seeds"][0] + 4)), "연속된 모델 seed 4개가 필요합니다."
	assert checkpoint["augmentations"] == ["square", "rectangle", "square_erasing", "square_perspective"]
	model = EfficientNetEnsemble(checkpoint["state_dicts"]).eval()
	n_params = sum(parameter.numel() for parameter in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	transform = build_valid_transform()
	files = load_images(args.input)
	rows = []
	with torch.inference_mode():
		for position in range(0, len(files), 32):
			batch_files = files[position:position + 32]
			images = []
			for path in batch_files:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images)
			member_probs = model.member_probabilities(inputs)
			if args.tta:
				member_probs = (member_probs + model.member_probabilities(inputs.flip(dims=[3]))) / 2
			probs = combine_probabilities(member_probs, args.temperature_rule).squeeze(1)
			assert bool(torch.isfinite(probs).all())
			for path, prob in zip(batch_files, probs):
				rows.append((path.stem, f"{prob.item():.6f}"))
	with Path(args.output).open("w", newline="", encoding="utf-8") as output:
		writer = csv.writer(output)
		writer.writerow(("id", "prob"))
		writer.writerows(rows)
	print(f"{args.output} 저장 완료 / {len(files)}장 / {n_params:,} parameters / TTA={args.tta}")


if __name__ == "__main__":
	main()


In [ ]:
package_dir = RESULT_DIR / 'package'
assert not package_dir.exists(), f'기존 패키지를 덮어쓰지 않습니다: {package_dir}'
package_dir.mkdir()
for source, destination in (
	(BASE_DIR / 'model.py', 'model.py'),
	(BASE_DIR / 'predict.py', 'predict.py'),
	(RESULT_DIR / 'model.pt', 'model.pt'),
	(BASE_DIR / 'baseline_new.ipynb', 'baseline_new.ipynb'),
):
	shutil.copy2(source, package_dir / destination)
package_bytes = sum(path.stat().st_size for path in package_dir.iterdir() if path.is_file())
assert package_bytes <= 200_000_000, f'패키지 크기 초과: {package_bytes:,}'
print('패키지 폴더:', package_dir)
print(f'전체 크기: {package_bytes:,} bytes / 압축 생성 없음')


### 패키지 자가 검증 (제출 전 필수!)

운영진 검증과 같은 방식으로, 패키지 폴더 안에서 `predict.py`를 CPU로 실행해 봅니다. 여기서 에러가 나면 검증도 실패합니다. Colab 런타임을 CPU로 두고 실행하면 10분 기준도 함께 확인할 수 있습니다.

이 검사는 package 폴더의 기본 명령으로 500장·ID·확률·600초·재현 차이를 확인합니다. 로컬 CPU 시간은 실제 Colab CPU 시간과 구분하며, Colab CPU 제한은 그 환경에서 실행해 확인해야 합니다.


In [ ]:
started = time.perf_counter()
check_path = RESULT_DIR / 'package_check.csv'
process = subprocess.run(
	[sys.executable, '-B', 'predict.py', '--input', str(BASE_DIR / 'test'), '--output', str(check_path)],
	cwd=package_dir, capture_output=True, text=True, encoding='utf-8', timeout=600,
	env={**os.environ, 'PYTHONIOENCODING': 'utf-8'},
)
seconds = time.perf_counter() - started
print(process.stdout)
assert process.returncode == 0, process.stderr
assert seconds <= 600, f'CPU 추론 600초 초과: {seconds:.2f}'


def read_probs(path):
	with path.open(newline='', encoding='utf-8-sig') as source:
		reader = csv.DictReader(source)
		assert reader.fieldnames == ['id', 'prob']
		rows = list(reader)
	assert (len(rows) == 500) and (len({row['id'] for row in rows}) == 500)
	assert {row['id'] for row in rows} == expected_ids
	assert all(np.isfinite(float(row['prob'])) and (0 <= float(row['prob']) <= 1) for row in rows)
	return {row['id']: float(row['prob']) for row in rows}


original_probs = read_probs(RESULT_DIR / 'submission.csv')
reproduced_probs = read_probs(check_path)
max_diff = max(abs(original_probs[image_id] - reproduced_probs[image_id]) for image_id in expected_ids)
assert max_diff <= 0.00001, f'CSV 재현 차이: {max_diff}'
print(f'CPU {seconds:.2f}s / CSV 최대 차이 {max_diff:.8f} / 재현 통과')


## 제출 전 체크리스트

- [ ] `submission.csv`가 `id,prob` 형식이고 테스트 500장 전부를 포함하는가?
- [ ] 파라미터 카운터 셀에서 **20M 이하**를 확인했는가? (앙상블·사전학습 백본 합산)
- [ ] 패키지 자가 검증 셀이 에러 없이 통과했는가? (Colab CPU 에서 500장 10분 이내)
- [ ] `predict.py`가 네트워크 없이 동작하는가? (다운로드 코드 금지, 가중치는 패키지에 동봉)
- [ ] 테스트 이미지를 학습에 사용하지 않았는가? (pseudo-labeling 포함 금지)
- [ ] 외부 이미지를 학습에 사용하지 않았는가?

## 개선 아이디어

- 채널 수·블록 수 조정, 학습률 스케줄러(CosineAnnealing 등) 추가
- 20M 이내의 사전학습 백본(MobileNetV3, EfficientNet-B0 등) fine-tuning — 가중치를 패키지에 포함할 것
- 증강 다양화: 밝기·색감·회전·노이즈 — 처음 보는 촬영 조건에 대한 일반화
- 검증 전략 개선: k-fold 교차검증으로 점수 추정의 신뢰도 높이기
- 과확신 완화: label smoothing, 확률 클리핑, 온도 조절(temperature scaling)

### 현재 제출본 추가 확인
- 최종 예측 모델은 B0 네 개 합산 16,035,316개 파라미터입니다.
- 학습용 사진은 제공 train 폴더의 학습 분할만 사용하며 테스트 데이터는 예측에만 사용합니다.
- 사전학습 가중치는 허용된 초기 가중치로만 사용하고, 추론 중 다운로드하지 않습니다.
- 학습 코드·추론 코드·전체 가중치를 함께 제출해야 합니다. 이 노트북은 패키지 폴더까지만 구성합니다.
- 제공된 실제 최종 가중치의 예측은 submission_submit.csv(Public 0.0066)와 대조할 수 있습니다.
- 같은 코드라도 다른 하드웨어·라이브러리에서 전체 재학습 결과의 비트 단위 일치까지 보장하지는 않습니다.
